In [0]:
METADATA_PATH = "s3://enterprise-lakehouse-data/metadata/pipeline_runs/"


GENERATE run_id (ONE PER PIPELINE RUN)

In [0]:
from datetime import datetime

run_id = f"BACKFILL-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S')}"


In [0]:
run_id

'BACKFILL-2026-01-30T07-17-46'

DEFINE STEP NAME

READ BRONZE DATA FROM S3

In [0]:

silver_df = spark.read.parquet("s3://enterprise-lakehouse-data/silver/maintenance_snapshots/")
curated_df = spark.read.parquet("s3://enterprise-lakehouse-data/silver_curated/maintenance_events/")
gold_df = spark.read.parquet("s3://enterprise-lakehouse-data/gold/fact_maintenance_daily/")


In [0]:

silver_count = silver_df.count()
curated_count = curated_df.count()
gold_count = gold_df.count()


In [0]:
print(silver_count)
print(curated_count)
print(gold_count)

2287
2287
201


Create metadata rows

In [0]:
from datetime import datetime

now = datetime.now()

metadata_rows = [
    (
        run_id,
        "silver_to_silver_curated",
        now, now,
        "SUCCESS",
        silver_count,
        curated_count,
        "BACKFILL"
    ),
    (
        run_id,
        "silver_curated_to_gold",
        now, now,
        "SUCCESS",
        curated_count,
        gold_count,
        "BACKFILL"
    )
]


Write metadata

In [0]:
columns = [
    "run_id",
    "step",
    "start_time",
    "end_time",
    "status",
    "records_read",
    "records_written",
    "error_message"
]

metadata_df = spark.createDataFrame(metadata_rows, columns)
metadata_df.write.mode("append").parquet(
    "s3://enterprise-lakehouse-data/metadata/pipeline_runs/"
)


In [0]:
dff1=spark.read.parquet("s3://enterprise-lakehouse-data/metadata/pipeline_runs/")

In [0]:
dff1.show(5)

+--------------------+--------------------+--------------------+--------------------+-------+------------+---------------+-------------+
|              run_id|                step|          start_time|            end_time| status|records_read|records_written|error_message|
+--------------------+--------------------+--------------------+--------------------+-------+------------+---------------+-------------+
|BACKFILL-2026-01-...|silver_to_silver_...|2026-01-30 07:43:...|2026-01-30 07:43:...|SUCCESS|        2287|           2287|     BACKFILL|
|BACKFILL-2026-01-...|silver_curated_to...|2026-01-30 07:43:...|2026-01-30 07:43:...|SUCCESS|        2287|            201|     BACKFILL|
+--------------------+--------------------+--------------------+--------------------+-------+------------+---------------+-------------+



In [0]:
dff1.display()

run_id,step,start_time,end_time,status,records_read,records_written,error_message
BACKFILL-2026-01-30T07-17-46,silver_to_silver_curated,2026-01-30T07:43:29.229Z,2026-01-30T07:43:29.229Z,SUCCESS,2287,2287,BACKFILL
BACKFILL-2026-01-30T07-17-46,silver_curated_to_gold,2026-01-30T07:43:29.229Z,2026-01-30T07:43:29.229Z,SUCCESS,2287,201,BACKFILL


DATA QUALITY METRICS

In [0]:
silver_df = spark.read.parquet(
    "s3://enterprise-lakehouse-data/silver/maintenance_snapshots/"
)

silver_curated_df = spark.read.parquet(
    "s3://enterprise-lakehouse-data/silver_curated/maintenance_events/"
)

quarantine_df = spark.read.parquet(
    "s3://enterprise-lakehouse-data/quarantine/maintenance_snapshots/"
)


In [0]:
silver_count = silver_df.count()
silver_curated_count = silver_curated_df.count()
quarantine_count = quarantine_df.count()


In [0]:
quarantine_df.printSchema()

root
 |-- maintenance_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- status: string (nullable = true)
 |-- actual_date: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- actual_date_dt: date (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- f_null_maintenance_id: boolean (nullable = true)
 |-- f_null_asset_id: boolean (nullable = true)
 |-- f_null_maintenance_type: boolean (nullable = true)
 |-- f_null_status: boolean (nullable = true)
 |-- f_negative_cost: boolean (nullable = true)
 |-- f_null_ingestion_ts: boolean (nullable = true)
 |-- failure_reason: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ingestion_date: date (nullable = true)



In [0]:
quarantine_df.select("failure_reason").show()

+---------------+
| failure_reason|
+---------------+
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
|[NEGATIVE_COST]|
+---------------+
only showing top 20 rows


In [0]:
negative_count_violation = quarantine_count


In [0]:
dq_pass_rate = silver_curated_count  / silver_count


Create DQ metrics record

In [0]:
from datetime import datetime

dq_row = [(
    run_id,
    "silver_to_silver_curated",
    datetime.now(),
    silver_count,
    silver_curated_count,
    quarantine_count,
    negative_count_violation,
    dq_pass_rate
)]
dq_columns = [
    "run_id",
    "step",
    "metric_time",
    "silver_count",
    "curated_count",
    "quarantine_count",
    "negative_count_violation",
    "dq_pass_rate"
]

dq_df = spark.createDataFrame(dq_row, dq_columns)


Write DQ metrics

In [0]:
dq_df.write.mode("append").parquet(
    "s3://enterprise-lakehouse-data/metadata/data_quality_metrics/"
)


In [0]:
silver_curated_count+quarantine_count

3821

In [0]:
silver_count

2287

LATE DATA & HISTORICAL CORRECTIONS

In [0]:
gold_scd_df = spark.read.parquet(
    "s3://enterprise-lakehouse-data/gold/dim_asset_scd/"
)
